In [1]:
print("hello")

hello


In [3]:
import os
from typing import TypedDict, Annotated

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


# --------------------------------------------------
# 1. Load environment variables
# --------------------------------------------------

load_dotenv()


# --------------------------------------------------
# 2. Create a tool
# --------------------------------------------------

@tool
def calculator(expression: str) -> str:
    """Calculate a basic mathematical expression."""

    try:
        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception:
        return "Could not calculate the expression."


# --------------------------------------------------
# 3. Create the LLM
# --------------------------------------------------

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)


# --------------------------------------------------
# 4. Bind tools to the LLM
# --------------------------------------------------

tools = [calculator]

llm_with_tools = llm.bind_tools(tools)


# --------------------------------------------------
# 5. Define graph state
# --------------------------------------------------

class State(TypedDict):
    messages: Annotated[
        list[BaseMessage],
        add_messages
    ]


# --------------------------------------------------
# 6. Agent node
# --------------------------------------------------

def agent_node(state: State):
    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }


# --------------------------------------------------
# 7. Tool node
# --------------------------------------------------

tool_node = ToolNode(tools)


# --------------------------------------------------
# 8. Decide where to go next
# --------------------------------------------------

def route_after_agent(state: State):
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END


# --------------------------------------------------
# 9. Build the graph
# --------------------------------------------------

builder = StateGraph(State)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {
        "tools": "tools",
        END: END
    }
)

builder.add_edge("tools", "agent")


# --------------------------------------------------
# 10. Compile the graph
# --------------------------------------------------

graph = builder.compile()


# --------------------------------------------------
# 11. Run the agent
# --------------------------------------------------

result = graph.invoke({
    "messages": [
        HumanMessage(
            content="What is 25 multiplied by 4? use the available tools"
        )
    ]
})


# --------------------------------------------------
# 12. Print the conversation
# --------------------------------------------------

for message in result["messages"]:

    print(type(message).__name__)
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    print("-" * 40)

HumanMessage
What is 25 multiplied by 4? use the available tools
----------------------------------------
AIMessage

Tool calls: [{'name': 'calculator', 'args': {'expression': '25 * 4'}, 'id': 'fc_1f071410-23ba-4bc4-a616-4b4f3d4727c1', 'type': 'tool_call'}]
----------------------------------------
ToolMessage
100
----------------------------------------
AIMessage
The product of 25 and 4 is **100**.
----------------------------------------


In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[
        list[BaseMessage],
        add_messages
    ]